# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Define Paths & Reset Dataset

In [ ]:
import os, shutil

BASE = "/content/drive/MyDrive/malware_project"

BENIGN_SRC = f"{BASE}/Benign"              # <-- CAPITAL B (important)
MALWARE_SRC = f"{BASE}/malimg_extracted"   # already extracted malware images
FINAL_DATASET = f"{BASE}/final_dataset"    # multi-class dataset

# Remove old dataset
shutil.rmtree(FINAL_DATASET, ignore_errors=True)
os.makedirs(FINAL_DATASET, exist_ok=True)

print("✔ Dataset folder reset")


✔ Dataset folder reset


# Binary → Image Conversion Function

In [ ]:
import numpy as np
from PIL import Image

def file_to_image(file_path, out_path, img_size=128):
    with open(file_path, "rb") as f:
        byte_data = f.read()

    arr = np.frombuffer(byte_data, dtype=np.uint8)
    dim = int(np.ceil(np.sqrt(len(arr))))

    padded = np.zeros(dim * dim, dtype=np.uint8)
    padded[:len(arr)] = arr

    img = padded.reshape(dim, dim)
    img = Image.fromarray(img, mode="L")
    img = img.resize((img_size, img_size))
    img.save(out_path)


# Benign File Type Mapping

In [ ]:
EXT_MAP = {
    ".exe": "benign_exe",
    ".dll": "benign_dll",
    ".mui": "benign_mui",
    ".mui.dll": "benign_mui"
}


# Convert Benign Files (Type-Aware)

In [ ]:
count = 0

for root, dirs, files in os.walk(BENIGN_SRC):
    for file in files:
        src = os.path.join(root, file)

        file_lower = file.lower()
        label = "benign_other"

        if file_lower.endswith(".exe"):
            label = "benign_exe"
        elif file_lower.endswith(".dll"):
            label = "benign_dll"
        elif file_lower.endswith(".mui") or file_lower.endswith(".mui.dll"):
            label = "benign_mui"

        out_dir = os.path.join(FINAL_DATASET, label)
        os.makedirs(out_dir, exist_ok=True)

        out_path = os.path.join(out_dir, f"{label}_{count}.png")

        try:
            file_to_image(src, out_path)
            count += 1
        except:
            pass

print("✔ Benign files processed:", count)


/tmp/ipython-input-3611399691.py:15: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(img, mode="L")


✔ Benign files processed: 9870


# Convert MALWARE Images (FAMILY-WISE)

In [ ]:
from PIL import Image

malware_total = 0

for family in os.listdir(MALWARE_SRC):
    family_path = os.path.join(MALWARE_SRC, family)

    if not os.path.isdir(family_path):
        continue

    label = "malware_" + family.lower().replace(".", "").replace(" ", "_")
    out_dir = os.path.join(FINAL_DATASET, label)
    os.makedirs(out_dir, exist_ok=True)

    i = 0
    for img_file in os.listdir(family_path):
        if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
            try:
                img = Image.open(os.path.join(family_path, img_file)).convert("L")
                img = img.resize((128, 128))
                img.save(os.path.join(out_dir, f"{label}_{i}.png"))
                i += 1
            except:
                pass

    malware_total += i
    print(f"✔ {label}: {i} images")

print("✔ Total malware images:", malware_total)


✔ malware_adialerc: 122 images
✔ malware_agentfyi: 116 images
✔ malware_allaplea: 2949 images
✔ malware_allaplel: 1591 images
✔ malware_aluerongen!j: 198 images
✔ malware_autorunk: 106 images
✔ malware_c2lopp: 146 images
✔ malware_c2lopgen!g: 200 images
✔ malware_dialplatformb: 177 images
✔ malware_dontovoa: 162 images
✔ malware_fakerean: 381 images
✔ malware_instantaccess: 431 images
✔ malware_lolydaaa1: 213 images
✔ malware_lolydaaa2: 184 images
✔ malware_lolydaaa3: 123 images
✔ malware_lolydaat: 159 images
✔ malware_malexgen!j: 136 images
✔ malware_obfuscatorad: 142 images
✔ malware_rbot!gen: 158 images
✔ malware_skintrimn: 80 images
✔ malware_swizzorgen!e: 128 images
✔ malware_swizzorgen!i: 132 images
✔ malware_vbat: 408 images
✔ malware_wintrimbx: 97 images
✔ malware_yunera: 800 images
✔ Total malware images: 9339


# Final Dataset Verification

In [ ]:
import os

FINAL_DATASET = "/content/drive/MyDrive/malware_project/final_dataset"

print("\nFINAL DATASET STRUCTURE:\n")

total_images = 0
benign_total = 0
malware_total = 0
last_type = None

for folder in sorted(os.listdir(FINAL_DATASET)):
    path = os.path.join(FINAL_DATASET, folder)
    if not os.path.isdir(path):
        continue

    count = len(os.listdir(path))
    total_images += count

    # Detect type
    if folder.startswith("benign"):
        benign_total += count
        current_type = "benign"
    else:
        malware_total += count
        current_type = "malware"

    # Add blank line between benign & malware
    if last_type and current_type != last_type:
        print()

    print(f"{folder:30s} -> {count:5d} images")
    last_type = current_type

print("\n" + "-" * 55)
print(f"TOTAL BENIGN IMAGES          -> {benign_total}")
print(f"TOTAL MALWARE IMAGES         -> {malware_total}")
print(f"TOTAL IMAGES (ALL)           -> {total_images}")
print("-" * 55)



FINAL DATASET STRUCTURE:

benign_dll                     ->  9784 images
benign_exe                     ->     7 images
benign_mui                     ->    75 images
benign_other                   ->     4 images

malware_adialerc               ->   122 images
malware_agentfyi               ->   116 images
malware_allaplea               ->  2949 images
malware_allaplel               ->  1591 images
malware_aluerongen!j           ->   198 images
malware_autorunk               ->   106 images
malware_c2lopgen!g             ->   200 images
malware_c2lopp                 ->   146 images
malware_dialplatformb          ->   177 images
malware_dontovoa               ->   162 images
malware_fakerean               ->   381 images
malware_instantaccess          ->   431 images
malware_lolydaaa1              ->   213 images
malware_lolydaaa2              ->   184 images
malware_lolydaaa3              ->   123 images
malware_lolydaat               ->   159 images
malware_malexgen!j             -